In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('src')
from config.paths import config
from config.constants import TARGET_COLUMN, NON_MEDICAL_FEATURES

In [10]:
df = pd.read_csv(config.cleaned_data_file)
print(f"Loaded dataset shape: {df.shape}")
print(f"Target distribution:\n{df[TARGET_COLUMN].value_counts()}")

Loaded dataset shape: (195196, 51)
Target distribution:
DEMENTED
0    137606
1     57590
Name: count, dtype: int64


In [11]:
print("DATA VALIDATION CHECKS")
print("-" * 40)

def validate_data(df):
    validation_results = {}

    validation_results['total_rows'] = len(df)
    validation_results['total_columns'] = len(df.columns)
    validation_results['missing_values'] = df.isnull().sum().sum()

    validation_results['duplicate_rows'] = df.duplicated().sum()

    validation_results['target_distribution'] = df[TARGET_COLUMN].value_counts().to_dict()

    numerical_cols = df.select_dtypes(include=[np.number]).columns
    validation_results['numerical_outliers'] = {}
    for col in numerical_cols:
        if col != TARGET_COLUMN:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            outliers = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
            validation_results['numerical_outliers'][col] = outliers

    categorical_cols = df.select_dtypes(include=['object']).columns
    validation_results['high_cardinality'] = {}
    for col in categorical_cols:
        if df[col].nunique() > 20:
            validation_results['high_cardinality'][col] = df[col].nunique()

    return validation_results

validation_before = validate_data(df)
print("Validation Results - Before Preprocessing:")
print(f"Total rows: {validation_before['total_rows']}")
print(f"Total columns: {validation_before['total_columns']}")
print(f"Missing values: {validation_before['missing_values']}")
print(f"Duplicate rows: {validation_before['duplicate_rows']}")
print(f"Target distribution: {validation_before['target_distribution']}")

if validation_before['duplicate_rows'] > 0:
    df = df.drop_duplicates()
    print(f"Removed {validation_before['duplicate_rows']} duplicate rows")

DATA VALIDATION CHECKS
----------------------------------------
Validation Results - Before Preprocessing:
Total rows: 195196
Total columns: 51
Missing values: 1772869
Duplicate rows: 0
Target distribution: {0: 137606, 1: 57590}


In [14]:
print("1. FEATURE SELECTION STRATEGY")
print("-" * 40)

medical_features_removed = [col for col in df.columns if col not in NON_MEDICAL_FEATURES and col != TARGET_COLUMN]
print(f"Medical features removed: {len(medical_features_removed)}")
for feature in medical_features_removed:
    print(f"  - {feature}: Excluded - medical feature")

df_selected = df[NON_MEDICAL_FEATURES + [TARGET_COLUMN]].copy()
print(f"After medical feature removal: {df_selected.shape}")

# Separate numerical and categorical features
numerical_features = df_selected.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df_selected.select_dtypes(include=['object']).columns.tolist()

# Remove target column if it's in the lists
if TARGET_COLUMN in numerical_features:
    numerical_features.remove(TARGET_COLUMN)
if TARGET_COLUMN in categorical_features:
    categorical_features.remove(TARGET_COLUMN)

print(f"Numerical features: {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")

# Handle high correlation only for numerical features
high_corr_pairs = []
features_to_remove = set()

if len(numerical_features) > 1:
    correlation_matrix = df_selected[numerical_features].corr()

    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            if abs(correlation_matrix.iloc[i, j]) > 0.8:
                high_corr_pairs.append((
                    correlation_matrix.columns[i],
                    correlation_matrix.columns[j],
                    correlation_matrix.iloc[i, j]
                ))

    for feat1, feat2, corr in high_corr_pairs:
        feat1_na = df_selected[feat1].isna().sum()
        feat2_na = df_selected[feat2].isna().sum()

        if feat1_na > feat2_na:
            features_to_remove.add(feat1)
            print(f"  - {feat1}: Removed - high correlation ({corr:.3f}) with {feat2}, more missing values")
        else:
            features_to_remove.add(feat2)
            print(f"  - {feat2}: Removed - high correlation ({corr:.3f}) with {feat1}, more missing values")

# Handle low variance for both numerical and categorical
low_variance_features = []
all_features = numerical_features + categorical_features

for col in all_features:
    if col not in features_to_remove:  # Don't check features already marked for removal
        unique_ratio = df_selected[col].nunique() / len(df_selected)
        if unique_ratio < 0.05:
            low_variance_features.append(col)
            print(f"  - {col}: Removed - low variance ({unique_ratio:.3f} unique ratio)")

# Remove all identified features
features_to_remove.update(low_variance_features)
df_selected = df_selected.drop(columns=features_to_remove)

# Update feature lists after removal
remaining_features = [col for col in df_selected.columns if col != TARGET_COLUMN]
numerical_features = [f for f in remaining_features if f in numerical_features]
categorical_features = [f for f in remaining_features if f in categorical_features]

print(f"Features after correlation and variance filtering: {len(remaining_features)}")

# Prepare data for Random Forest with proper handling of mixed data types
X_temp = df_selected.drop(columns=[TARGET_COLUMN])
y_temp = df_selected[TARGET_COLUMN]

# Handle missing values appropriately
X_processed = X_temp.copy()

# Fill numerical missing values with median
for col in numerical_features:
    if col in X_processed.columns and X_processed[col].isna().any():
        X_processed[col] = X_processed[col].fillna(X_processed[col].median())

# Fill categorical missing values with mode
for col in categorical_features:
    if col in X_processed.columns and X_processed[col].isna().any():
        X_processed[col] = X_processed[col].fillna(X_processed[col].mode().iloc[0] if not X_processed[col].mode().empty else 'Unknown')

# Convert categorical variables to numeric using Label Encoding
label_encoders = {}
for col in categorical_features:
    if col in X_processed.columns:
        le = LabelEncoder()
        X_processed[col] = le.fit_transform(X_processed[col].astype(str))
        label_encoders[col] = le

# Train Random Forest for feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_processed, y_temp)

feature_importance = pd.DataFrame({
    'feature': X_processed.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

top_features = feature_importance.head(20)['feature'].tolist()
print(f"Top 20 features selected by importance:")
for i, (_, row) in enumerate(feature_importance.head(20).iterrows(), 1):
    print(f"  {i:2d}. {row['feature']}: {row['importance']:.4f}")

df_final_selected = df_selected[top_features + [TARGET_COLUMN]]
print(f"Final selected features: {df_final_selected.shape}")

1. FEATURE SELECTION STRATEGY
----------------------------------------
Medical features removed: 0
After medical feature removal: (195196, 51)
Numerical features: 48
Categorical features: 2
  - FORMVER: Removed - high correlation (0.897) with VISITYR, more missing values
  - NACCVNUM: Removed - high correlation (0.969) with NACCFDYS, more missing values
  - NACCAGE: Removed - high correlation (-0.890) with BIRTHYR, more missing values
  - NACCAGEB: Removed - high correlation (-0.901) with BIRTHYR, more missing values
  - INHISP: Removed - high correlation (0.817) with HISPANIC, more missing values
  - RACETER: Removed - high correlation (0.804) with INRASEC, more missing values
  - INRATER: Removed - high correlation (0.900) with RACETER, more missing values
  - NACCAGE: Removed - high correlation (0.936) with NACCAGEB, more missing values
  - INVISITS: Removed - high correlation (0.924) with INLIVWTH, more missing values
  - INCALLS: Removed - high correlation (0.942) with INLIVWTH, m

In [15]:
print("2. MISSING DATA HANDLING")
print("-" * 40)

missing_before = df_final_selected.isnull().sum()
missing_features = missing_before[missing_before > 0]

print("Missing values before imputation:")
for feature, count in missing_features.items():
    percentage = (count / len(df_final_selected)) * 100
    print(f"  {feature}: {count} ({percentage:.2f}%)")

numerical_features = df_final_selected.select_dtypes(include=[np.number]).columns.tolist()
numerical_features = [f for f in numerical_features if f != TARGET_COLUMN and f in df_final_selected.columns]
categorical_features = df_final_selected.select_dtypes(include=['object']).columns.tolist()

imputation_strategy = {}

for feature in numerical_features:
    if df_final_selected[feature].isnull().sum() > 0:
        imputation_strategy[feature] = 'median'
        median_val = df_final_selected[feature].median()
        df_final_selected[f'{feature}_missing'] = df_final_selected[feature].isnull().astype(int)
        df_final_selected[feature].fillna(median_val, inplace=True)
        print(f"  {feature}: Median imputation ({median_val:.2f}) + missing indicator")

for feature in categorical_features:
    if df_final_selected[feature].isnull().sum() > 0:
        imputation_strategy[feature] = 'mode_unknown'
        mode_val = df_final_selected[feature].mode()[0] if not df_final_selected[feature].mode().empty else 'Unknown'
        df_final_selected[feature].fillna('Unknown', inplace=True)
        print(f"  {feature}: Mode imputation ('{mode_val}') + 'Unknown' category")

missing_after = df_final_selected.isnull().sum().sum()
print(f"Missing values after imputation: {missing_after}")

2. MISSING DATA HANDLING
----------------------------------------
Missing values before imputation:
Missing values after imputation: 0


In [16]:
print("3. FEATURE CREATION")
print("-" * 40)

if 'BIRTHYR' in df_final_selected.columns and 'VISITYR' in df_final_selected.columns:
    df_final_selected['calculated_age'] = df_final_selected['VISITYR'] - df_final_selected['BIRTHYR']
    print("Created: calculated_age")

if 'EDUC' in df_final_selected.columns:
    df_final_selected['education_level'] = pd.cut(df_final_selected['EDUC'],
                                          bins=[0, 12, 16, 20, 30],
                                          labels=['High_School', 'Bachelors', 'Masters', 'Doctorate'])
    print("Created: education_level - binned into 4 levels")

if 'NACCAGE' in df_final_selected.columns:
    df_final_selected['age_group'] = pd.cut(df_final_selected['NACCAGE'],
                                    bins=[0, 65, 75, 85, 120],
                                    labels=['Young_Senior', 'Middle_Senior', 'Old_Senior', 'Elderly'])
    print("Created: age_group - binned into 4 groups")

cognitive_components = []
if 'EDUC' in df_final_selected.columns:
    normalized_educ = (df_final_selected['EDUC'] - df_final_selected['EDUC'].min()) / (df_final_selected['EDUC'].max() - df_final_selected['EDUC'].min())
    cognitive_components.append(normalized_educ)
    print("Added: Education to cognitive reserve score")

if 'PRIMLANG' in df_final_selected.columns:
    english_speaker = (df_final_selected['PRIMLANG'] == 1).astype(int)
    cognitive_components.append(english_speaker * 0.5)
    print("Added: English proficiency to cognitive reserve score")

if cognitive_components:
    cognitive_reserve_score = sum(cognitive_components) / len(cognitive_components)
    df_final_selected['cognitive_reserve_score'] = cognitive_reserve_score
    print("Created: cognitive_reserve_score - composite of education and language")

lifestyle_components = []
if 'MARISTAT' in df_final_selected.columns:
    married_indicator = df_final_selected['MARISTAT'].isin([1, 6]).astype(int)
    lifestyle_components.append(married_indicator)
    print("Added: Marital status to lifestyle risk index")

if 'NACCLIVS' in df_final_selected.columns:
    living_alone_indicator = (df_final_selected['NACCLIVS'] == 1).astype(int)
    lifestyle_components.append(living_alone_indicator * 0.5)
    print("Added: Living alone to lifestyle risk index")

if 'RESIDENC' in df_final_selected.columns:
    institutional_living = df_final_selected['RESIDENC'].isin([3, 4]).astype(int)
    lifestyle_components.append(institutional_living * 0.7)
    print("Added: Institutional living to lifestyle risk index")

if lifestyle_components:
    lifestyle_risk_index = sum(lifestyle_components) / len(lifestyle_components)
    df_final_selected['lifestyle_risk_index'] = lifestyle_risk_index
    print("Created: lifestyle_risk_index - composite of social factors")

social_components = []
if 'INVISITS' in df_final_selected.columns:
    frequent_visits = (df_final_selected['INVISITS'] <= 3).astype(int)
    social_components.append(frequent_visits)
    print("Added: Frequent visits to social engagement index")

if 'INCALLS' in df_final_selected.columns:
    frequent_calls = (df_final_selected['INCALLS'] <= 3).astype(int)
    social_components.append(frequent_calls)
    print("Added: Frequent calls to social engagement index")

if 'INLIVWTH' in df_final_selected.columns:
    lives_with_coparticipant = (df_final_selected['INLIVWTH'] == 1).astype(int)
    social_components.append(lives_with_coparticipant)
    print("Added: Living with co-participant to social engagement index")

if social_components:
    social_engagement_index = sum(social_components) / len(social_components)
    df_final_selected['social_engagement_index'] = social_engagement_index
    print("Created: social_engagement_index - composite of social interactions")

support_components = []
if 'INRELTO' in df_final_selected.columns:
    immediate_family = df_final_selected['INRELTO'].isin([1, 2, 3]).astype(int)
    support_components.append(immediate_family)
    print("Added: Immediate family to social support index")

if 'INRELY' in df_final_selected.columns:
    reliable_coparticipant = (df_final_selected['INRELY'] == 0).astype(int)
    support_components.append(reliable_coparticipant)
    print("Added: Reliable co-participant to social support index")

if support_components:
    social_support_index = sum(support_components) / len(support_components)
    df_final_selected['social_support_index'] = social_support_index
    print("Created: social_support_index - composite of support network quality")

if 'EDUC' in df_final_selected.columns and 'social_engagement_index' in df_final_selected.columns:
    df_final_selected['education_social_interaction'] = df_final_selected['EDUC'] * df_final_selected['social_engagement_index']
    print("Created: education_social_interaction - education × social engagement")

if 'NACCAGE' in df_final_selected.columns:
    df_final_selected['age_squared'] = df_final_selected['NACCAGE'] ** 2
    print("Created: age_squared - polynomial feature for non-linear relationships")

if 'EDUC' in df_final_selected.columns:
    df_final_selected['education_squared'] = df_final_selected['EDUC'] ** 2
    print("Created: education_squared - polynomial feature for non-linear relationships")

print(f"After feature creation: {df_final_selected.shape}")

3. FEATURE CREATION
----------------------------------------
After feature creation: (195196, 2)


In [17]:
print("4. ENCODING & SCALING")
print("-" * 40)

categorical_features = df_final_selected.select_dtypes(include=['object', 'category']).columns.tolist()

low_cardinality_features = []
high_cardinality_features = []

for feature in categorical_features:
    if df_final_selected[feature].nunique() <= 10:
        low_cardinality_features.append(feature)
    else:
        high_cardinality_features.append(feature)

print(f"Low-cardinality features for one-hot encoding: {len(low_cardinality_features)}")
print(f"High-cardinality features for target encoding: {len(high_cardinality_features)}")

df_encoded = df_final_selected.copy()

for feature in low_cardinality_features:
    dummies = pd.get_dummies(df_encoded[feature], prefix=feature, drop_first=True)
    df_encoded = pd.concat([df_encoded, dummies], axis=1)
    df_encoded.drop(columns=[feature], inplace=True)
    print(f"One-hot encoded: {feature} -> {len(dummies.columns)} new features")

label_encoder = LabelEncoder()
for feature in high_cardinality_features:
    df_encoded[f'{feature}_encoded'] = label_encoder.fit_transform(df_encoded[feature].astype(str))
    df_encoded.drop(columns=[feature], inplace=True)
    print(f"Label encoded: {feature}")

numerical_features_for_scaling = df_encoded.select_dtypes(include=[np.number]).columns.tolist()
numerical_features_for_scaling = [f for f in numerical_features_for_scaling if f != TARGET_COLUMN]

print(f"Scaling {len(numerical_features_for_scaling)} numerical features using StandardScaler")
print("Justification: StandardScaler standardizes features to mean=0, variance=1, essential for distance-based algorithms and gradient descent optimization")

scaler = StandardScaler()
scaled_features = scaler.fit_transform(df_encoded[numerical_features_for_scaling])
scaled_df = pd.DataFrame(scaled_features, columns=[f'{feat}_scaled' for feat in numerical_features_for_scaling])

df_final = pd.concat([df_encoded.drop(columns=numerical_features_for_scaling), scaled_df], axis=1)

preprocessing_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

print(f"After encoding and scaling: {df_final.shape}")

4. ENCODING & SCALING
----------------------------------------
Low-cardinality features for one-hot encoding: 0
High-cardinality features for target encoding: 1
Label encoded: NACCID
Scaling 1 numerical features using StandardScaler
Justification: StandardScaler standardizes features to mean=0, variance=1, essential for distance-based algorithms and gradient descent optimization
After encoding and scaling: (195196, 2)
